In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
PyTorch 机器信息一键速查
> python torch_sysinfo.py
"""
%pip install torch psutil


  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
   ---------------------------------------- 0.0/111.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/111.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/111.0 MB 3.0 MB/s eta 0:00:38
    --------------------------------------- 2.4/111.0 MB 4.2 MB/s eta 0:00:26
   - -------------------------------------- 3.1/111.0 MB 4.0 MB/s eta 0:00:27
   - -------------------------------------- 3.9/111.0 MB 4.2 MB/s eta 0:00:26
   - -------------------------------------- 5.2/111.0 MB 4.6 MB/s eta 0:00:24
   -- ------------------------------------- 6.3/111.0 MB 4.8 MB/s eta 0:00:22
   -- ------------------------------------- 8.1/111.0 MB 5.1 MB/s eta 0:00:21
   ---- ----------------------------------- 12.1/111.0 MB 6.8 MB/s eta 0:00:15
   ------ --------------------------------- 17.0/111.0 MB 8.5 MB/s eta 0:00:12
   -------- ------------------------------- 23.1/111.0 MB 10.6 MB/s eta 0:00:09
   -----

In [3]:
import os
import sys
import platform
import subprocess
import torch
import psutil   # pip install psutil

# ---------- 颜色工具 ----------
class Color:
    HEADER = "\033[95m"
    OKBLUE = "\033[94m"
    OKCYAN = "\033[96m"
    OKGREEN = "\033[92m"
    WARNING = "\033[93m"
    FAIL = "\033[91m"
    ENDC = "\033[0m"
    BOLD = "\033[1m"

def cprint(text, color=Color.OKGREEN):
    print(color + text + Color.ENDC)

# ---------- 软件栈 ----------
cprint("==== Software Stack ====", Color.HEADER)
print(f"Python     : {platform.python_version()}")
print(f"PyTorch    : {torch.__version__}")
print(f"CUDA Build : {torch.version.cuda}")
print(f"cuDNN      : {torch.backends.cudnn.version()}")

# ---------- CPU ----------
cprint("==== CPU ====", Color.HEADER)
cpu = platform.processor() or "Unknown"
cores = psutil.cpu_count(logical=False)
threads = psutil.cpu_count(logical=True)
print(f"CPU        : {cpu}")
print(f"Cores      : {cores}")
print(f"Threads    : {threads}")

# ---------- GPU ----------
cprint("==== GPU ====", Color.HEADER)
if not torch.cuda.is_available():
    cprint("No GPU detected.", Color.FAIL)
else:
    n_gpu = torch.cuda.device_count()
    print(f"GPU Count  : {n_gpu}")
    for i in range(n_gpu):
        prop = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}  : {prop.name}")
        print(f"         Memory : {prop.total_memory // 1024**2} MB")
        print(f"         Compute: {prop.major}.{prop.minor}")
    # 实时利用率
    cprint("---- Real-Time Utilization ----", Color.OKCYAN)
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=True
        )
        for line in result.stdout.strip().split("\n"):
            idx, name, used, total, util = line.split(", ")
            print(f"  GPU {idx}: {util}% | Memory {used}/{total} MB")
    except FileNotFoundError:
        cprint("nvidia-smi not found; skip real-time util.", Color.WARNING)

# ---------- 分布式环境 ----------
cprint("==== Distributed / Env ====", Color.HEADER)
dist_env_keys = ["RANK", "LOCAL_RANK", "WORLD_SIZE",
                 "MASTER_ADDR", "MASTER_PORT", "CUDA_VISIBLE_DEVICES"]
for k in dist_env_keys:
    print(f"{k:20} : {os.environ.get(k, 'Not set')}")

# ---------- 随机种子 ----------
cprint("==== Reproducibility ====", Color.HEADER)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
print("Random seed set to 42 for CPU & all GPUs.")

# ---------- 设备选择 ----------
cprint("==== Default Device ====", Color.HEADER)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.rand(1, 1).to(device)
print(f"Default tensor placed on : {x.device}")

==== Software Stack ====
Python     : 3.10.19
PyTorch    : 2.9.1+cpu
CUDA Build : None
cuDNN      : None
==== CPU ====
CPU        : AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD
Cores      : 6
Threads    : 12
==== GPU ====
No GPU detected.
==== Distributed / Env ====
RANK                 : Not set
LOCAL_RANK           : Not set
WORLD_SIZE           : Not set
MASTER_ADDR          : Not set
MASTER_PORT          : Not set
CUDA_VISIBLE_DEVICES : Not set
==== Reproducibility ====
Random seed set to 42 for CPU & all GPUs.
==== Default Device ====
Default tensor placed on : cpu


#####  Check GPU hardware 

In [5]:
!nvidia-smi

Sat Dec  6 12:37:36 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 561.17                 Driver Version: 561.17         CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   40C    P8             11W /  120W |       0MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

#####  Check GPU hardware 

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Wed_Oct_30_01:18:48_Pacific_Daylight_Time_2024
Cuda compilation tools, release 12.6, V12.6.85
Build cuda_12.6.r12.6/compiler.35059454_0


In [ ]:
# !pip install torch==1.7.1+cu110 torchvision==0.8.2+cu110 torchaudio==0.7.2 -f https://download.pytorch.org/whl/torch_stable.html